In [ ]:
# !pip install git+https://github.com/monash-emu/summer3wip.git

In [ ]:
import numpy as np
from plotly import graph_objects as go

import pandas as pd
pd.options.plotting.backend = "plotly"

from summer3.graph import defer, Parameter, CompartmentValues
from summer3.epi import Stratification, CompartmentMap, \
    ManagedArray, CategoryData, TransitionFlow, CompartmentalEpiModel, \
    build_istate, dti_to_epoch

In [ ]:
population = 1e6  
seed = 1.0
start_time = 0
end_time = 50
model_comps = [
    "susceptible_vacc",
    "susceptible_unvacc", 
    "infectious", 
    "recovered",
]
infect_comps = ["infectious"]
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)
epi_model = CompartmentalEpiModel(humans, range(start_time, end_time))


def iprocess(compartment_values: ManagedArray, contact_rate: float, infectious_compartments: tuple):
    n_infectious = compartment_values.query(infectious_compartments).sum()
    n_population = compartment_values.sum()
    infectious_prevalence = n_infectious / n_population
    return contact_rate * infectious_prevalence

recovery = TransitionFlow("recovery", disease_state["infectious"], disease_state["recovered"], Parameter("recovery_rate", 0.0))
epi_model.add_flow(recovery)
base_parameters = {"effective_contact_rate": 1.2, "recovery_rate": 0.2}

In [ ]:
for vacc_status in ["vacc", "unvacc"]:
    vacc_effect = Parameter("vacc_effect", 0.0) if vacc_status == "vacc" else 0.0
    contact_rate = Parameter("effective_contact_rate", 0.0) * (1.0 - vacc_effect)
    foi = defer(iprocess)(CompartmentValues, contact_rate, disease_state["infectious"])
    infection = TransitionFlow(
        f"infection_{vacc_status}",
        disease_state[f"susceptible_{vacc_status}"], 
        disease_state["infectious"], 
        foi,
    )
    epi_model.add_flow(infection)

In [ ]:
vacc_prop = 0.5
start_pop = [
    (population - seed) * vacc_prop,
    (population - seed) * (1.0 - vacc_prop),
    seed, 
    0.0,
]
epi_model.set_initial_population(CategoryData(disease_state.categories(), np.array(start_pop)))

In [ ]:
vacc_effects = [0.0, 0.5]
outputs = pd.DataFrame(columns=vacc_effects)
base_parameters = {"effective_contact_rate": 1.2, "recovery_rate": 0.2}

for effect in vacc_effects:
    results = epi_model.run(base_parameters | {"vacc_effect": effect})
    outputs[effect] = results["compartments"].to_pandas_df()["infectious"]

In [ ]:
fig = go.Figure()
legend_format = {"title": "vaccination effect"}
xaxis_format = {"title": "days"}
for c in outputs.columns:
    colour = f"rgba({c}, 0, {1.0 - c}, 1)"
    fig.add_trace(go.Scatter(x=outputs.index, y=outputs[c], mode="lines", name=round(c, 1), line={"color": colour}))
fig.update_layout(title="effect of vaccination on the epidemic", legend=legend_format, xaxis=xaxis_format, yaxis={"title": "infection prevalence"})